In [23]:
%pip -q install pandas numpy plotly scipy requests

import os
import re
import json
import zipfile
import warnings
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
from scipy import stats
from IPython.display import display, Markdown
from IPython.display import HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [24]:
REPO_ZIP = (
    "https://github.com/Koustav41/"
    "Phonpe-Digital-Payments/archive/refs/heads/main.zip"
)

WORK_DIR = Path("/content/phonepe_project")
OUTPUT_DIR = Path("/content/phonepe_outputs")
ZIP_PATH = Path("/content/phonepe_repo.zip")

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("PHONEPE DIGITAL PAYMENTS ANALYSIS")
print("=" * 70)
print("Downloading repository...")

response = requests.get(REPO_ZIP, timeout=120)
response.raise_for_status()
ZIP_PATH.write_bytes(response.content)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(WORK_DIR)

csv_match = list(WORK_DIR.rglob("aggregated_transaction.csv"))

if not csv_match:
    raise FileNotFoundError(
        "Could not locate aggregated_transaction.csv. "
        "Check that the GitHub repository is accessible."
    )

DATA_DIR = csv_match[0].parent
print("Data directory:", DATA_DIR)
print("Download and extraction completed.")

PHONEPE DIGITAL PAYMENTS ANALYSIS
Data directory: /content/phonepe_project/Phonpe-Digital-Payments-main/data
Download and extraction completed.


**LOAD ALL NINE DATASETS**

In [25]:
required_files = [
    "aggregated_transaction.csv",
    "aggregated_user.csv",
    "aggregated_merchant.csv",
    "map_transaction.csv",
    "map_user.csv",
    "map_merchant.csv",
    "top_transaction.csv",
    "top_user.csv",
    "top_merchant.csv",
]

missing_files = [
    name for name in required_files
    if not (DATA_DIR / name).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing expected files: {missing_files}"
    )

data = {
    Path(filename).stem: pd.read_csv(DATA_DIR / filename)
    for filename in required_files
}

print("\nDATASET DIMENSIONS")
for name, df in data.items():
    print(f"{name:28s} {df.shape[0]:>8,} rows x {df.shape[1]} columns")

agg_tx = data["aggregated_transaction"].copy()
agg_user = data["aggregated_user"].copy()
agg_merchant = data["aggregated_merchant"].copy()
map_tx = data["map_transaction"].copy()
map_user = data["map_user"].copy()
map_merchant = data["map_merchant"].copy()
top_tx = data["top_transaction"].copy()
top_user = data["top_user"].copy()
top_merchant = data["top_merchant"].copy()


DATASET DIMENSIONS
aggregated_transaction          3,671 rows x 5 columns
aggregated_user                 1,224 rows x 4 columns
aggregated_merchant             1,178 rows x 4 columns
map_transaction                26,616 rows x 6 columns
map_user                       26,622 rows x 5 columns
map_merchant                   24,737 rows x 5 columns
top_transaction                10,200 rows x 6 columns
top_user                       10,200 rows x 6 columns
top_merchant                    9,732 rows x 6 columns


**GENERAL DATA QUALITY AUDIT**

In [26]:
audit_rows = []
for name,df in data.items():
    audit_rows.append({
       "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_percentage": round(
            df.isna().sum().sum() /
            max(df.size, 1) * 100, 3
        ),
    })
    audit = pd.DataFrame(audit_rows)
    print("\nDATA QUALITY SUMMARY")
    display(audit)
    audit.to_csv(OUTPUT_DIR / "data_quality_audit.csv", index=False)
    print("\n Column Names")
    for name,df in data.items():
        print(f"{name}:{list(df.columns)}")



DATA QUALITY SUMMARY


,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,aggregated_transaction,3671,5,0,0,0.0



 Column Names
aggregated_transaction:['state', 'year', 'quarter', 'transaction_type', 'count']
aggregated_user:['state', 'year', 'quarter', 'registered_count']
aggregated_merchant:['state', 'year', 'quarter', 'registered_count']
map_transaction:['state', 'year', 'quarter', 'district', 'count', 'amount']
map_user:['state', 'year', 'quarter', 'district', 'registered_count']
map_merchant:['state', 'year', 'quarter', 'district', 'registered_count']
top_transaction:['state', 'year', 'quarter', 'level', 'entity', 'count']
top_user:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']
top_merchant:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']

DATA QUALITY SUMMARY


,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,aggregated_transaction,3671,5,0,0,0.0
1,aggregated_user,1224,4,0,0,0.0



 Column Names
aggregated_transaction:['state', 'year', 'quarter', 'transaction_type', 'count']
aggregated_user:['state', 'year', 'quarter', 'registered_count']
aggregated_merchant:['state', 'year', 'quarter', 'registered_count']
map_transaction:['state', 'year', 'quarter', 'district', 'count', 'amount']
map_user:['state', 'year', 'quarter', 'district', 'registered_count']
map_merchant:['state', 'year', 'quarter', 'district', 'registered_count']
top_transaction:['state', 'year', 'quarter', 'level', 'entity', 'count']
top_user:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']
top_merchant:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']

DATA QUALITY SUMMARY


...


 Column Names
aggregated_transaction:['state', 'year', 'quarter', 'transaction_type', 'count']
aggregated_user:['state', 'year', 'quarter', 'registered_count']
aggregated_merchant:['state', 'year', 'quarter', 'registered_count']
map_transaction:['state', 'year', 'quarter', 'district', 'count', 'amount']
map_user:['state', 'year', 'quarter', 'district', 'registered_count']
map_merchant:['state', 'year', 'quarter', 'district', 'registered_count']
top_transaction:['state', 'year', 'quarter', 'level', 'entity', 'count']
top_user:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']
top_merchant:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']

DATA QUALITY SUMMARY


,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,aggregated_transaction,3671,5,0,0,0.0
1,aggregated_user,1224,4,0,0,0.0
2,aggregated_merchant,1178,4,0,0,0.0
3,map_transaction,26616,6,0,0,0.0
4,map_user,26622,5,0,0,0.0
5,map_merchant,24737,5,0,0,0.0
6,top_transaction,10200,6,0,0,0.0
7,top_user,10200,6,0,0,0.0



 Column Names
aggregated_transaction:['state', 'year', 'quarter', 'transaction_type', 'count']
aggregated_user:['state', 'year', 'quarter', 'registered_count']
aggregated_merchant:['state', 'year', 'quarter', 'registered_count']
map_transaction:['state', 'year', 'quarter', 'district', 'count', 'amount']
map_user:['state', 'year', 'quarter', 'district', 'registered_count']
map_merchant:['state', 'year', 'quarter', 'district', 'registered_count']
top_transaction:['state', 'year', 'quarter', 'level', 'entity', 'count']
top_user:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']
top_merchant:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']

DATA QUALITY SUMMARY


,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,aggregated_transaction,3671,5,0,0,0.0
1,aggregated_user,1224,4,0,0,0.0
2,aggregated_merchant,1178,4,0,0,0.0
3,map_transaction,26616,6,0,0,0.0
4,map_user,26622,5,0,0,0.0
5,map_merchant,24737,5,0,0,0.0
6,top_transaction,10200,6,0,0,0.0
7,top_user,10200,6,0,0,0.0
8,top_merchant,9732,6,0,0,0.0



 Column Names
aggregated_transaction:['state', 'year', 'quarter', 'transaction_type', 'count']
aggregated_user:['state', 'year', 'quarter', 'registered_count']
aggregated_merchant:['state', 'year', 'quarter', 'registered_count']
map_transaction:['state', 'year', 'quarter', 'district', 'count', 'amount']
map_user:['state', 'year', 'quarter', 'district', 'registered_count']
map_merchant:['state', 'year', 'quarter', 'district', 'registered_count']
top_transaction:['state', 'year', 'quarter', 'level', 'entity', 'count']
top_user:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']
top_merchant:['state', 'year', 'quarter', 'level', 'entity', 'registered_count']


****

**CLEANING AND STANDARDIZATION**

In [27]:

def clean_state(value):
    if pd.isna(value):
        return np.nan
    return re.sub(r"\s+", " ", str(value).replace("-", " ").strip()).title()

def clean_district(value):
    if pd.isna(value):
        return np.nan
    value = re.sub(r"\s+", " ", str(value).strip())
    value = re.sub(r"\s+district$", "", value, flags=re.I)
    return value.title()

def prepare_period(df):
    df = df.copy()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["quarter"] = pd.to_numeric(df["quarter"], errors="coerce")
    df = df.dropna(subset=["year", "quarter"]).copy()
    df["year"] = df["year"].astype(int)
    df["quarter"] = df["quarter"].astype(int)
    df = df[df["quarter"].between(1, 4)].copy()
    df["period"] = (
        df["year"].astype(str)
        + " Q"
        + df["quarter"].astype(str)
    )
    df["period_sort"] = df["year"] * 10 + df["quarter"]
    return df

for key in data:
    data[key] = prepare_period(data[key])

agg_tx = data["aggregated_transaction"].copy()
agg_user = data["aggregated_user"].copy()
agg_merchant = data["aggregated_merchant"].copy()
map_tx = data["map_transaction"].copy()
map_user = data["map_user"].copy()
map_merchant = data["map_merchant"].copy()
top_tx = data["top_transaction"].copy()
top_user = data["top_user"].copy()
top_merchant = data["top_merchant"].copy()

for df in [
    agg_tx, agg_user, agg_merchant,
    map_tx, map_user, map_merchant,
    top_tx, top_user, top_merchant
]:
    if "state" in df.columns:
        df["state"] = df["state"].apply(clean_state)
    if "district" in df.columns:
        df["district"] = df["district"].apply(clean_district)
    if "entity" in df.columns:
        df["entity"] = df["entity"].apply(clean_district)


numeric_columns = {
    "aggregated_transaction": ["count"],
    "aggregated_user": ["registered_count"],
    "aggregated_merchant": ["registered_count"],
    "map_transaction": ["count", "amount"],
    "map_user": ["registered_count"],
    "map_merchant": ["registered_count"],
    "top_transaction": ["count"],
    "top_user": ["registered_count"],
    "top_merchant": ["registered_count"],
}

for name, cols in numeric_columns.items():
    for col in cols:
        if col in data[name].columns:
            data[name][col] = pd.to_numeric(
                data[name][col], errors="coerce"
            )


agg_tx = data["aggregated_transaction"]
agg_user = data["aggregated_user"]
agg_merchant = data["aggregated_merchant"]
map_tx = data["map_transaction"]
map_user = data["map_user"]
map_merchant = data["map_merchant"]
top_tx = data["top_transaction"]
top_user = data["top_user"]
top_merchant = data["top_merchant"]


for name in data:
    data[name] = data[name].drop_duplicates().copy()

agg_tx = data["aggregated_transaction"]
agg_user = data["aggregated_user"]
agg_merchant = data["aggregated_merchant"]
map_tx = data["map_transaction"]
map_user = data["map_user"]
map_merchant = data["map_merchant"]
top_tx = data["top_transaction"]
top_user = data["top_user"]
top_merchant = data["top_merchant"]

print("\nCleaning completed.")
print("Negative transaction counts:",
      int((map_tx["count"].dropna() < 0).sum()))
print("Negative transaction amounts:",
      int((map_tx["amount"].dropna() < 0).sum()))


Cleaning completed.
Negative transaction counts: 0
Negative transaction amounts: 0


**TRANSACTION METRICS**

In [28]:
map_tx = map_tx.dropna(subset=["year", "quarter"]).copy()
map_tx["count"] = pd.to_numeric(map_tx["count"], errors="coerce")
map_tx["amount"] = pd.to_numeric(map_tx["amount"], errors="coerce")

valid_tx = map_tx.dropna(subset=["count", "amount"]).copy()
valid_tx = valid_tx[
    (valid_tx["count"] >= 0) & (valid_tx["amount"] >= 0)
].copy()

quarterly_tx = (
    valid_tx.groupby(["year", "quarter"], as_index=False)
    .agg(
        transaction_count=("count", "sum"),
        transaction_value=("amount", "sum")
    )
)

quarterly_tx["period"] = (
    quarterly_tx["year"].astype(str)
    + " Q"
    + quarterly_tx["quarter"].astype(str)
)
quarterly_tx["period_sort"] = (
    quarterly_tx["year"] * 10 + quarterly_tx["quarter"]
)
quarterly_tx["average_transaction_value"] = (
    quarterly_tx["transaction_value"]
    / quarterly_tx["transaction_count"].replace(0, np.nan)
)
quarterly_tx = quarterly_tx.sort_values("period_sort")

yearly_tx = (
    quarterly_tx.groupby("year", as_index=False)
    .agg(
        transaction_count=("transaction_count", "sum"),
        transaction_value=("transaction_value", "sum")
    )
)

yearly_tx["average_transaction_value"] = (
    yearly_tx["transaction_value"]
    / yearly_tx["transaction_count"].replace(0, np.nan)
)

yearly_tx["value_lakh_crore"] = (
    yearly_tx["transaction_value"] / 1e12
)

yearly_tx["count_billion"] = (
    yearly_tx["transaction_count"] / 1e9
)

yearly_tx["value_yoy_pct"] = (
    yearly_tx["transaction_value"].pct_change() * 100
)
yearly_tx["count_yoy_pct"] = (
    yearly_tx["transaction_count"].pct_change() * 100
)

# Full-year analysis only: 2018-2025.
full_years = yearly_tx[
    yearly_tx["year"].between(2018, 2025)
].copy()

def calculate_cagr(start_value, end_value, periods):
    if (
        pd.isna(start_value) or pd.isna(end_value)
        or start_value <= 0 or end_value <= 0
        or periods <= 0
    ):
        return np.nan
    return ((end_value / start_value) ** (1 / periods) - 1) * 100

cagr_value = np.nan
cagr_count = np.nan

if len(full_years) >= 2:
    first = full_years.iloc[0]
    last = full_years.iloc[-1]
    elapsed_years = int(last["year"] - first["year"])

    cagr_value = calculate_cagr(
        first["transaction_value"],
        last["transaction_value"],
        elapsed_years
    )
    cagr_count = calculate_cagr(
        first["transaction_count"],
        last["transaction_count"],
        elapsed_years
    )

print("\nYEARLY TRANSACTION SUMMARY")
display(yearly_tx.round(2))

print(f"\nValue CAGR (first to last available full year): {cagr_value:.2f}%")
print(f"Count CAGR (first to last available full year): {cagr_count:.2f}%")

yearly_tx.to_csv(OUTPUT_DIR / "yearly_transaction_summary.csv", index=False)
quarterly_tx.to_csv(OUTPUT_DIR / "quarterly_transaction_summary.csv", index=False)


YEARLY TRANSACTION SUMMARY


,year,transaction_count,transaction_value,average_transaction_value,value_lakh_crore,count_billion,value_yoy_pct,count_yoy_pct
0,2018,1109043733,1.627120e+12,1467.14,1.63,1.11,NaN,NaN
1,2019,4124423596,6.288070e+12,1524.59,6.29,4.12,286.45,271.89
2,2020,8022666617,1.465420e+13,1826.60,14.65,8.02,133.05,94.52
3,2021,18453525820,3.308903e+13,1793.10,33.09,18.45,125.80,130.02
4,2022,38640727109,6.296631e+13,1629.53,62.97,38.64,90.29,109.39
5,2023,64295972092,9.460256e+13,1471.36,94.60,64.30,50.24,66.39
6,2024,99333478997,1.297122e+14,1305.83,129.71,99.33,37.11,54.49
7,2025,128544039967,1.553767e+14,1208.74,155.38,128.54,19.79,29.41
8,2026,74915984776,8.894252e+13,1187.23,88.94,74.92,-42.76,-41.72



Value CAGR (first to last available full year): 91.80%
Count CAGR (first to last available full year): 97.18%


**PAYMENT TYPE ANALYSIS**

In [29]:
agg_tx["count"] = pd.to_numeric(agg_tx["count"], errors="coerce")
agg_tx = agg_tx.dropna(subset=["count", "transaction_type"]).copy()
agg_tx = agg_tx[agg_tx["count"] >= 0].copy()

type_year = (
    agg_tx.groupby(["year", "transaction_type"], as_index=False)
    .agg(transaction_count=("count", "sum"))
)

type_year["total_count"] = (
    type_year.groupby("year")["transaction_count"].transform("sum")
)
type_year["share_pct"] = (
    type_year["transaction_count"]
    / type_year["total_count"].replace(0, np.nan) * 100
)

print("\nPAYMENT TYPE SHARE BY YEAR")
display(type_year.tail(18).round(2))

type_year.to_csv(OUTPUT_DIR / "payment_type_by_year.csv", index=False)

fig = px.area(
    type_year.sort_values("year"),
    x="year",
    y="share_pct",
    color="transaction_type",
    groupnorm="percent",
    title="Payment-Type Share of Transaction Count",
    labels={
        "year": "Year",
        "share_pct": "Share (%)",
        "transaction_type": "Payment type"
    }
)
fig.show()


PAYMENT TYPE SHARE BY YEAR


,year,transaction_type,transaction_count,total_count,share_pct
9,2021,P2P,7765367854,18453525820,42.08
10,2021,Retail,7945462671,18453525820,43.06
11,2021,Utility,2742695295,18453525820,14.86
12,2022,P2P,14701456015,38640727109,38.05
13,2022,Retail,20131833810,38640727109,52.10
14,2022,Utility,3807437284,38640727109,9.85
15,2023,P2P,22744323623,64295972092,35.37
16,2023,Retail,36277246271,64295972092,56.42
17,2023,Utility,5274402198,64295972092,8.20
18,2024,P2P,33201235118,99333478997,33.42


**STATE-LEVEL ANALYSIS**

In [30]:
state_tx = (
    map_tx.groupby("state", as_index=False)
    .agg(
        transaction_count=("count", "sum"),
        transaction_value=("amount", "sum")
    )
)
state_tx["average_transaction_value"] = (
    state_tx["transaction_value"]
    / state_tx["transaction_count"].replace(0, np.nan)
)
state_tx["value_lakh_crore"] = state_tx["transaction_value"] / 1e12
state_tx = state_tx.sort_values("transaction_value", ascending=False)

print("\nTOP 10 STATES BY TRANSACTION VALUE — ALL AVAILABLE QUARTERS")
display(state_tx.head(10).round(2))

state_tx.to_csv(OUTPUT_DIR / "state_transaction_summary.csv", index=False)

fig = px.bar(
    state_tx.head(10).sort_values("transaction_value"),
    x="transaction_value",
    y="state",
    orientation="h",
    title="Top 10 States by Transaction Value — All Available Quarters",
    labels={
        "transaction_value": "Transaction value (₹)",
        "state": "State"
    }
)
fig.show()
if len(full_years):
    latest_full_year = int(full_years["year"].max())

    state_latest = (
        map_tx[map_tx["year"] == latest_full_year]
        .groupby("state", as_index=False)
        .agg(
            transaction_count=("count", "sum"),
            transaction_value=("amount", "sum")
        )
        .sort_values("transaction_value", ascending=False)
    )

    print(f"\nTOP 10 STATES IN {latest_full_year}")
    display(state_latest.head(10).round(2))
    state_latest.to_csv(
        OUTPUT_DIR / "top_states_latest_full_year.csv",
        index=False
    )

    fig = px.bar(
        state_latest.head(10).sort_values("transaction_value"),
        x="transaction_value",
        y="state",
        orientation="h",
        title=f"Top 10 States by Transaction Value ({latest_full_year})",
        labels={
            "transaction_value": "Transaction value (₹)",
            "state": "State"
        }
    )
    fig.show()


TOP 10 STATES BY TRANSACTION VALUE — ALL AVAILABLE QUARTERS


,state,transaction_count,transaction_value,average_transaction_value,value_lakh_crore
15,karnataka,55399289713,6.930963e+13,1251.09,69.31
20,maharashtra,58090476182,6.805764e+13,1171.58,68.06
31,telangana,44799513185,6.585734e+13,1470.05,65.86
1,andhra-pradesh,37210943664,6.193168e+13,1664.34,61.93
33,uttar-pradesh,37648808330,4.878129e+13,1295.69,48.78
28,rajasthan,30719527575,4.355004e+13,1417.67,43.55
4,bihar,22417600473,3.192985e+13,1424.32,31.93
19,madhya-pradesh,25481760471,3.175480e+13,1246.18,31.75
35,west-bengal,18471647241,2.769694e+13,1499.43,27.70
25,odisha,17062129019,2.132037e+13,1249.57,21.32



TOP 10 STATES IN 2025


,state,transaction_count,transaction_value
20,maharashtra,16864743344,1.825195e+13
15,karnataka,15377209002,1.795170e+13
31,telangana,12716111919,1.691960e+13
1,andhra-pradesh,10516843692,1.526312e+13
33,uttar-pradesh,11694702997,1.357051e+13
28,rajasthan,8898900728,1.141928e+13
4,bihar,7151040418,9.013142e+12
19,madhya-pradesh,7642175093,8.631933e+12
35,west-bengal,5845209194,7.913366e+12
25,odisha,5147629406,5.674281e+12


**DISTRICT-LEVEL ANALYSIS**

In [31]:
district_tx = (
    map_tx.groupby(["state", "district"], as_index=False)
    .agg(
        transaction_count=("count", "sum"),
        transaction_value=("amount", "sum")
    )
)
district_tx["average_transaction_value"] = (
    district_tx["transaction_value"]
    / district_tx["transaction_count"].replace(0, np.nan)
)
district_tx = district_tx.sort_values(
    "transaction_value", ascending=False
)

print("\nTOP 15 DISTRICTS BY TRANSACTION VALUE — ALL AVAILABLE QUARTERS")
display(district_tx.head(15).round(2))

district_tx.to_csv(
    OUTPUT_DIR / "district_transaction_summary.csv",
    index=False
)

fig = px.bar(
    district_tx.head(15).sort_values("transaction_value"),
    x="transaction_value",
    y="district",
    color="state",
    orientation="h",
    title="Top 15 Districts by Transaction Value",
    labels={
        "transaction_value": "Transaction value (₹)",
        "district": "District"
    }
)
fig.show()


TOP 15 DISTRICTS BY TRANSACTION VALUE — ALL AVAILABLE QUARTERS


,state,district,transaction_count,transaction_value,average_transaction_value
299,karnataka,bengaluru urban district,28298372145,3.133106e+13,1107.17
422,maharashtra,pune district,14676040673,1.366115e+13,930.85
656,telangana,rangareddy district,9786588903,1.316399e+13,1345.11
647,telangana,medchal malkajgiri district,9219780560,1.223491e+13,1327.03
565,rajasthan,jaipur district,7983979964,1.027488e+13,1286.94
634,telangana,hyderabad district,7281737302,9.439999e+12,1296.39
25,andhra-pradesh,visakhapatnam district,3814667794,5.498809e+12,1441.49
419,maharashtra,nashik district,3738510317,4.797116e+12,1283.16
429,maharashtra,thane district,4232336996,4.498927e+12,1062.99
219,haryana,gurugram district,4710101089,4.384427e+12,930.86


**USER AND MERCHANT TRENDS**

In [32]:
def yearly_registration(df, label):
    if "registered_count" not in df.columns:
        return pd.DataFrame(columns=["year", label])

    temp = df.dropna(subset=["registered_count"]).copy()
    temp["registered_count"] = pd.to_numeric(
        temp["registered_count"], errors="coerce"
    )
    temp = temp.dropna(subset=["registered_count"])
    result = (
        temp.groupby("year", as_index=False)
        .agg(**{label: ("registered_count", "sum")})
    )
    return result

users_year = yearly_registration(agg_user, "registered_users")
merchants_year = yearly_registration(
    agg_merchant, "registered_merchants"
)

registrations = pd.merge(
    users_year, merchants_year,
    on="year", how="outer"
).sort_values("year")

print("\nSTATE-REGISTRATION TOTALS SUMMED BY YEAR")
display(registrations.round(0))

registrations.to_csv(
    OUTPUT_DIR / "yearly_user_merchant_summary.csv",
    index=False
)

fig = go.Figure()

if "registered_users" in registrations.columns:
    fig.add_trace(go.Scatter(
        x=registrations["year"],
        y=registrations["registered_users"],
        mode="lines+markers",
        name="Registered users"
    ))

if "registered_merchants" in registrations.columns:
    fig.add_trace(go.Scatter(
        x=registrations["year"],
        y=registrations["registered_merchants"],
        mode="lines+markers",
        name="Registered merchants"
    ))

fig.update_layout(
    title="Reported State-Level User and Merchant Registrations",
    xaxis_title="Year",
    yaxis_title="Sum of state-level reported counts",
    yaxis_type="log"
)
fig.show()

# State-level users-per-merchant ratio for the latest common full year.
common_years = set(agg_user["year"].unique()) & set(
    agg_merchant["year"].unique()
)
common_full_years = sorted(
    int(y) for y in common_years
    if 2018 <= int(y) <= 2025
)

if common_full_years:
    ratio_year = common_full_years[-1]

    u = (
        agg_user[agg_user["year"] == ratio_year]
        .groupby("state", as_index=False)["registered_count"]
        .sum()
        .rename(columns={"registered_count": "registered_users"})
    )
    m = (
        agg_merchant[agg_merchant["year"] == ratio_year]
        .groupby("state", as_index=False)["registered_count"]
        .sum()
        .rename(columns={"registered_count": "registered_merchants"})
    )

    user_merchant = u.merge(m, on="state", how="inner")
    user_merchant["users_per_merchant"] = (
        user_merchant["registered_users"]
        / user_merchant["registered_merchants"].replace(0, np.nan)
    )
    user_merchant = user_merchant.sort_values(
        "users_per_merchant", ascending=False
    )

    print(f"\nUSERS PER MERCHANT BY STATE — {ratio_year}")
    display(user_merchant.head(15).round(2))

    user_merchant.to_csv(
        OUTPUT_DIR / "users_per_merchant_by_state.csv",
        index=False
    )

    fig = px.bar(
        user_merchant.head(15).sort_values("users_per_merchant"),
        x="users_per_merchant",
        y="state",
        orientation="h",
        title=f"States with Highest Users per Merchant ({ratio_year})",
        labels={
            "users_per_merchant": "Users per merchant",
            "state": "State"
        }
    )
    fig.show()


STATE-REGISTRATION TOTALS SUMMED BY YEAR


,year,registered_users,registered_merchants
0,2018,295463493,1093223
1,2019,608263755,24089863
2,2020,929811386,49409155
3,2021,1274896586,87434840
4,2022,1618542119,128261654
5,2023,1935311656,145609082
6,2024,2269047870,166243282
7,2025,2590198844,186729048
8,2026,1406221018,100347879



USERS PER MERCHANT BY STATE — 2025


,state,registered_users,registered_merchants,users_per_merchant
17,ladakh,889992,10087,88.23
18,lakshadweep,65159,1275,51.11
22,meghalaya,2758471,57500,47.97
29,sikkim,1496944,32946,45.44
0,andaman-&-nicobar-islands,799908,19087,41.91
24,nagaland,2094349,58541,35.78
9,goa,4945425,138857,35.62
21,manipur,2947160,95680,30.80
2,arunachal-pradesh,3276393,110537,29.64
7,dadra-&-nagar-haveli-&-daman-&-diu,2488452,101554,24.50


**COVID-PERIOD COMPARISON**

In [33]:
covid_periods = quarterly_tx[
    quarterly_tx["period"].isin(["2020 Q1", "2020 Q2", "2020 Q3", "2020 Q4"])
].copy()

print("\n2020 QUARTERLY TRANSACTION VALUE")
display(covid_periods[
    ["period", "transaction_count", "transaction_value",
     "average_transaction_value"]
].round(2))

if len(covid_periods) >= 2:
    fig = px.line(
        covid_periods,
        x="period",
        y="transaction_value",
        markers=True,
        title="Transaction Value During 2020",
        labels={
            "period": "Quarter",
            "transaction_value": "Transaction value (₹)"
        }
    )
    fig.show()



2020 QUARTERLY TRANSACTION VALUE


,period,transaction_count,transaction_value,average_transaction_value
8,2020 Q1,1632700391,2.700008e+12,1653.71
9,2020 Q2,1460073590,2.648633e+12,1814.04
10,2020 Q3,2048019938,3.905743e+12,1907.08
11,2020 Q4,2881872698,5.399819e+12,1873.72


**QUARTERLY SEASONALITY**

In [34]:
seasonality = quarterly_tx.copy()
annual_totals = (
    seasonality.groupby("year")["transaction_value"]
    .transform("sum")
)
seasonality["annual_value_share_pct"] = (
    seasonality["transaction_value"]
    / annual_totals.replace(0, np.nan) * 100
)

seasonality_full = seasonality[
    seasonality["year"].between(2021, 2025)
].copy()

quarter_shares = (
    seasonality_full.groupby("quarter", as_index=False)
    .agg(
        mean_share_pct=("annual_value_share_pct", "mean"),
        std_share_pct=("annual_value_share_pct", "std"),
        observations=("annual_value_share_pct", "count")
    )
)

print("\nAVERAGE QUARTER SHARE OF ANNUAL TRANSACTION VALUE (2021-2025)")
display(quarter_shares.round(2))

quarter_shares.to_csv(
    OUTPUT_DIR / "quarterly_seasonality.csv",
    index=False
)

fig = px.bar(
    quarter_shares,
    x="quarter",
    y="mean_share_pct",
    error_y="std_share_pct",
    title="Average Share of Annual Transaction Value by Quarter (2021-2025)",
    labels={
        "quarter": "Quarter",
        "mean_share_pct": "Mean annual share (%)"
    }
)
fig.show()

# Friedman test across years, treating each year as a block.
season_wide = seasonality_full.pivot(
    index="year",
    columns="quarter",
    values="annual_value_share_pct"
).dropna()

friedman_p = np.nan
friedman_stat = np.nan

if season_wide.shape[0] >= 3 and all(
    q in season_wide.columns for q in [1, 2, 3, 4]
):
    friedman = stats.friedmanchisquare(
        season_wide[1],
        season_wide[2],
        season_wide[3],
        season_wide[4]
    )
    friedman_stat = float(friedman.statistic)
    friedman_p = float(friedman.pvalue)

print("Friedman seasonality test statistic:", friedman_stat)
print("Friedman seasonality test p-value:", friedman_p)



AVERAGE QUARTER SHARE OF ANNUAL TRANSACTION VALUE (2021-2025)


,quarter,mean_share_pct,std_share_pct,observations
0,1,21.59,1.95,5
1,2,24.02,0.84,5
2,3,25.82,1.33,5
3,4,28.57,1.45,5


Friedman seasonality test statistic: 14.039999999999992
Friedman seasonality test p-value: 0.0028512084304522568


**STATE TOTAL RECONCILIATION CHECK**

In [35]:
agg_state_q = (
    agg_tx.groupby(["state", "year", "quarter"], as_index=False)
    .agg(aggregated_category_count=("count", "sum"))
)

map_state_q = (
    map_tx.groupby(["state", "year", "quarter"], as_index=False)
    .agg(district_count_sum=("count", "sum"))
)

reconciliation = agg_state_q.merge(
    map_state_q,
    on=["state", "year", "quarter"],
    how="outer",
    indicator=True
)

reconciliation["difference"] = (
    reconciliation["aggregated_category_count"]
    - reconciliation["district_count_sum"]
)
reconciliation["absolute_difference"] = reconciliation["difference"].abs()

print("\nSTATE-QUARTER COUNT RECONCILIATION")
print("Matched state-quarters:",
      int((reconciliation["_merge"] == "both").sum()))
print("Unmatched state-quarters:",
      int((reconciliation["_merge"] != "both").sum()))
print("Exact matches:",
      int((reconciliation["difference"] == 0).sum()))
print("Nonzero differences:",
      int((reconciliation["difference"].fillna(0) != 0).sum()))

display(reconciliation.sort_values(
    "absolute_difference", ascending=False
).head(10))

reconciliation.to_csv(
    OUTPUT_DIR / "state_quarter_reconciliation.csv",
    index=False
)


STATE-QUARTER COUNT RECONCILIATION
Matched state-quarters: 1224
Unmatched state-quarters: 0
Exact matches: 1224
Nonzero differences: 0


,state,year,quarter,aggregated_category_count,district_count_sum,_merge,difference,absolute_difference
1223,west-bengal,2026,2,1782353009,1782353009,both,0,0
0,andaman-&-nicobar-islands,2018,1,9089,9089,both,0,0
1,andaman-&-nicobar-islands,2018,2,15062,15062,both,0,0
2,andaman-&-nicobar-islands,2018,3,21154,21154,both,0,0
3,andaman-&-nicobar-islands,2018,4,31139,31139,both,0,0
4,andaman-&-nicobar-islands,2019,1,40726,40726,both,0,0
5,andaman-&-nicobar-islands,2019,2,46764,46764,both,0,0
6,andaman-&-nicobar-islands,2019,3,40623,40623,both,0,0
1207,west-bengal,2022,2,298813188,298813188,both,0,0
1206,west-bengal,2022,1,244077613,244077613,both,0,0


**STATISTICAL TEST: FULL-YEAR GROWTH**

In [36]:
annual_growth = full_years["value_yoy_pct"].dropna()

growth_t_stat = np.nan
growth_p_value = np.nan

if len(annual_growth) >= 2:
    growth_test = stats.ttest_1samp(annual_growth, popmean=0)
    growth_t_stat = float(growth_test.statistic)
    growth_p_value = float(growth_test.pvalue)

print("\nYEAR-OVER-YEAR VALUE GROWTH TEST")
print("Observed yearly growth rates:", annual_growth.round(2).tolist())
print("One-sample t statistic:", growth_t_stat)
print("Two-sided p-value:", growth_p_value)


YEAR-OVER-YEAR VALUE GROWTH TEST
Observed yearly growth rates: [286.45, 133.05, 125.8, 90.29, 50.24, 37.11, 19.79]
One-sample t statistic: 3.0994577296044983
Two-sided p-value: 0.02113109283712541


**CORRELATION: STATE TRANSACTION VALUE**

In [37]:
corr_df = state_tx.dropna(
    subset=["transaction_value", "transaction_count"]
).copy()

pearson_r = np.nan
pearson_p = np.nan
spearman_rho = np.nan
spearman_p = np.nan

if len(corr_df) >= 3:
    pearson = stats.pearsonr(
        corr_df["transaction_count"],
        corr_df["transaction_value"]
    )
    spearman = stats.spearmanr(
        corr_df["transaction_count"],
        corr_df["transaction_value"]
    )
    pearson_r, pearson_p = map(float, pearson)
    spearman_rho, spearman_p = map(float, spearman)

print("\nSTATE-LEVEL COUNT/VALUE CORRELATION")
print("Pearson r:", pearson_r, "p-value:", pearson_p)
print("Spearman rho:", spearman_rho, "p-value:", spearman_p)


STATE-LEVEL COUNT/VALUE CORRELATION
Pearson r: 0.9902396432645658 p-value: 1.0948051161106092e-30
Spearman rho: 0.9935649935649937 p-value: 9.434183869132177e-34


**TRANSACTION VALUE TREND CHARTS**

In [38]:
fig = px.line(
    yearly_tx,
    x="year",
    y="value_lakh_crore",
    markers=True,
    title="Yearly Transaction Value",
    labels={
        "year": "Year",
        "value_lakh_crore": "Transaction value (₹ lakh crore)"
    }
)
fig.show()

fig = px.line(
    quarterly_tx,
    x="period_sort",
    y="transaction_count",
    markers=True,
    title="Quarterly Transaction Count",
    labels={
        "period_sort": "Quarter (ordered)",
        "transaction_count": "Transaction count"
    }
)
fig.update_xaxes(
    tickmode="array",
    tickvals=quarterly_tx["period_sort"][::4],
    ticktext=quarterly_tx["period"][::4]
)
fig.show()

fig = px.line(
    yearly_tx,
    x="year",
    y="average_transaction_value",
    markers=True,
    title="Average Transaction Value by Year",
    labels={
        "year": "Year",
        "average_transaction_value": "Average transaction value (₹)"
    }
)
fig.show()

**AUTOMATIC FINDINGS SUMMARY**

In [39]:
findings = []

if len(yearly_tx):
    first_year = yearly_tx.iloc[0]
    last_year = yearly_tx.iloc[-1]

    findings.append(
        f"- The earliest year in the dataset is {int(first_year['year'])}; "
        f"the latest available year is {int(last_year['year'])}."
    )
    findings.append(
        f"- The earliest-year transaction value in the district-level data "
        f"is approximately ₹{first_year['value_lakh_crore']:.2f} lakh crore."
    )
    findings.append(
        f"- The latest-year transaction value in the district-level data "
        f"is approximately ₹{last_year['value_lakh_crore']:.2f} lakh crore. "
        f"Check whether this is a partial year before comparing it to full years."
    )

if np.isfinite(cagr_value):
    findings.append(
        f"- Transaction-value CAGR across the first and last available "
        f"full years from 2018–2025 is {cagr_value:.2f}%."
    )

if np.isfinite(cagr_count):
    findings.append(
        f"- Transaction-count CAGR across the same full-year period is "
        f"{cagr_count:.2f}%."
    )

if len(state_tx):
    top_state = state_tx.iloc[0]
    findings.append(
        f"- By total transaction value across the available period, "
        f"the leading state in this dataset is {top_state['state']}."
    )

if len(district_tx):
    top_district = district_tx.iloc[0]
    findings.append(
        f"- By total district-level transaction value across the available "
        f"period, the leading district is {top_district['district']} "
        f"({top_district['state']})."
    )

if len(type_year):
    latest_type_year = int(type_year["year"].max())
    latest_types = type_year[type_year["year"] == latest_type_year]
    leading_type = latest_types.sort_values(
        "share_pct", ascending=False
    ).iloc[0]

    findings.append(
        f"- In the latest year represented in the payment-type dataset "
        f"({latest_type_year}), the largest transaction-count category is "
        f"{leading_type['transaction_type']} "
        f"({leading_type['share_pct']:.1f}% of category counts)."
    )

if np.isfinite(pearson_r):
    findings.append(
        f"- Across states, transaction count and transaction value have "
        f"a Pearson correlation of {pearson_r:.3f} "
        f"(p={pearson_p:.4g}). This is an association, not evidence of causation."
    )

if np.isfinite(friedman_p):
    findings.append(
        f"- The Friedman test for quarter-level annual-value shares "
        f"gave p={friedman_p:.4g}. Interpret this cautiously because "
        f"the number of complete years is small."
    )

if np.isfinite(growth_p_value):
    findings.append(
        f"- A one-sample t-test of yearly value growth rates against zero "
        f"gave p={growth_p_value:.4g}. This test has few annual observations "
        f"and should be treated as exploratory."
    )
print(findings)

['- The earliest year in the dataset is 2018; the latest available year is 2026.', '- The earliest-year transaction value in the district-level data is approximately ₹1.63 lakh crore.', '- The latest-year transaction value in the district-level data is approximately ₹88.94 lakh crore. Check whether this is a partial year before comparing it to full years.', '- Transaction-value CAGR across the first and last available full years from 2018–2025 is 91.80%.', '- Transaction-count CAGR across the same full-year period is 97.18%.', '- By total transaction value across the available period, the leading state in this dataset is karnataka.', '- By total district-level transaction value across the available period, the leading district is bengaluru urban district (karnataka).', '- In the latest year represented in the payment-type dataset (2026), the largest transaction-count category is Retail (63.2% of category counts).', '- Across states, transaction count and transaction value have a Pearso

**Reports**


In [40]:
report_lines = [
    "# PhonePe Digital Payments Analysis",
    "",
    "## 1. Project objective",
    "",
    "Analyze publicly available, aggregated PhonePe Pulse data to understand "
    "digital payment growth, payment-type mix, geographic concentration, "
    "average transaction value, and state-level user/merchant patterns.",
    "",
    "## 2. Data and methodology",
    "",
    f"- Source repository: https://github.com/Koustav41/Phonpe-Digital-Payments",
    f"- Dataset directory: `{DATA_DIR}`",
    f"- Number of CSV datasets loaded: {len(data)}",
    "- Transaction count and value summaries use district-level map transaction data.",
    "- Payment-type shares use the aggregated transaction dataset.",
    "- Exact duplicate rows were removed; numeric fields were parsed and checked.",
    "- CAGR is calculated only across available full years within 2018–2025.",
    "",
    "## 3. Key findings calculated from the data",
    "",
]
report_lines.extend(findings if findings else [
    "- No findings could be calculated. Check whether the input data is empty."
])

report_lines.extend([
    "",
    "## 4. Statistical outputs",
    "",
    f"- Transaction value CAGR: {cagr_value:.2f}%",
    f"- Transaction count CAGR: {cagr_count:.2f}%",
    f"- Growth-rate t-test p-value: {growth_p_value}",
    f"- Quarter-share Friedman test p-value: {friedman_p}",
    f"- State count/value Pearson correlation: {pearson_r}",
    f"- State count/value correlation p-value: {pearson_p}",
    "",
    "## 5. Limitations",
    "",
    "- 2026 includes only Q1 and Q2 in the repository profile; it is not a full-year total.",
    "- Registered-user and registered-merchant values are cumulative state-level counts. "
    "Summing them across states can double-count entities that appear in more than one state.",
    "- District-level and aggregated payment-category datasets have different grains. "
    "Reconciliation differences should be investigated rather than silently corrected.",
    "- Statistical significance does not demonstrate causation.",
    "- The dataset is aggregated and anonymized; it does not support conclusions about "
    "individual users or their personal transaction histories.",
    "- Results depend on the data release and its metric definitions.",
    "",
    "## 6. Recommendations",
    "",
    "- Prioritize states and districts using both transaction value and transaction count.",
    "- Compare payment-type shares over time to identify changes in the mix of use cases.",
    "- Investigate high users-per-merchant ratios before recommending merchant onboarding.",
    "- Validate data reconciliation and coverage before presenting results as definitive.",
    "- Build a Streamlit dashboard only after the notebook calculations have been checked.",
    "",
    "## 7. Output files",
    "",
    "- `data_quality_audit.csv`",
    "- `yearly_transaction_summary.csv`",
    "- `quarterly_transaction_summary.csv`",
    "- `payment_type_by_year.csv`",
    "- `state_transaction_summary.csv`",
    "- `district_transaction_summary.csv`",
    "- `yearly_user_merchant_summary.csv`",
    "- `quarterly_seasonality.csv`",
    "- `state_quarter_reconciliation.csv`",
    "- Additional CSVs for rankings and ratios when data is available.",
    "",
    "Report generated automatically by the notebook."
])

report_text = "\n".join(report_lines)
report_path = OUTPUT_DIR / "PhonePe_Analysis_Report.md"
report_path.write_text(report_text, encoding="utf-8")

display(Markdown(report_text))

# PhonePe Digital Payments Analysis

## 1. Project objective

Analyze publicly available, aggregated PhonePe Pulse data to understand digital payment growth, payment-type mix, geographic concentration, average transaction value, and state-level user/merchant patterns.

## 2. Data and methodology

- Source repository: https://github.com/Koustav41/Phonpe-Digital-Payments
- Dataset directory: `/content/phonepe_project/Phonpe-Digital-Payments-main/data`
- Number of CSV datasets loaded: 9
- Transaction count and value summaries use district-level map transaction data.
- Payment-type shares use the aggregated transaction dataset.
- Exact duplicate rows were removed; numeric fields were parsed and checked.
- CAGR is calculated only across available full years within 2018–2025.

## 3. Key findings calculated from the data

- The earliest year in the dataset is 2018; the latest available year is 2026.
- The earliest-year transaction value in the district-level data is approximately ₹1.63 lakh crore.
- The latest-year transaction value in the district-level data is approximately ₹88.94 lakh crore. Check whether this is a partial year before comparing it to full years.
- Transaction-value CAGR across the first and last available full years from 2018–2025 is 91.80%.
- Transaction-count CAGR across the same full-year period is 97.18%.
- By total transaction value across the available period, the leading state in this dataset is karnataka.
- By total district-level transaction value across the available period, the leading district is bengaluru urban district (karnataka).
- In the latest year represented in the payment-type dataset (2026), the largest transaction-count category is Retail (63.2% of category counts).
- Across states, transaction count and transaction value have a Pearson correlation of 0.990 (p=1.095e-30). This is an association, not evidence of causation.
- The Friedman test for quarter-level annual-value shares gave p=0.002851. Interpret this cautiously because the number of complete years is small.
- A one-sample t-test of yearly value growth rates against zero gave p=0.02113. This test has few annual observations and should be treated as exploratory.

## 4. Statistical outputs

- Transaction value CAGR: 91.80%
- Transaction count CAGR: 97.18%
- Growth-rate t-test p-value: 0.02113109283712541
- Quarter-share Friedman test p-value: 0.0028512084304522568
- State count/value Pearson correlation: 0.9902396432645658
- State count/value correlation p-value: 1.0948051161106092e-30

## 5. Limitations

- 2026 includes only Q1 and Q2 in the repository profile; it is not a full-year total.
- Registered-user and registered-merchant values are cumulative state-level counts. Summing them across states can double-count entities that appear in more than one state.
- District-level and aggregated payment-category datasets have different grains. Reconciliation differences should be investigated rather than silently corrected.
- Statistical significance does not demonstrate causation.
- The dataset is aggregated and anonymized; it does not support conclusions about individual users or their personal transaction histories.
- Results depend on the data release and its metric definitions.

## 6. Recommendations

- Prioritize states and districts using both transaction value and transaction count.
- Compare payment-type shares over time to identify changes in the mix of use cases.
- Investigate high users-per-merchant ratios before recommending merchant onboarding.
- Validate data reconciliation and coverage before presenting results as definitive.
- Build a Streamlit dashboard only after the notebook calculations have been checked.

## 7. Output files

- `data_quality_audit.csv`
- `yearly_transaction_summary.csv`
- `quarterly_transaction_summary.csv`
- `payment_type_by_year.csv`
- `state_transaction_summary.csv`
- `district_transaction_summary.csv`
- `yearly_user_merchant_summary.csv`
- `quarterly_seasonality.csv`
- `state_quarter_reconciliation.csv`
- Additional CSVs for rankings and ratios when data is available.

Report generated automatically by the notebook.